In [20]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from sklearn.ensemble import RandomForestRegressor
from sklearn.decomposition import PCA
import joblib
import warnings

warnings.filterwarnings("ignore", category=RuntimeWarning)

In [21]:
PATH = r"E:\satdata\Custom\yucatan_sst_chlor_cdomslope_ssl_bathy_2010-2019_v3.pkl"
MODEL_PATH = r"F:\Programming\GitHub\nasa-murep\yucatan\experiments\isolation_forest\models\iforest_pipeline4.joblib"

In [22]:
# with open(PATH, "rb") as f:
#     blob = pickle.load(f)

# df = blob["data"]
# metadata = blob["metadata"]
# df = df[(df.index.year >= 2010) & (df.index.year <= 2019)]

# grid_shape = np.asarray(df["sst"].iloc[0]).shape

# def shape_ok(a):
#     return a is not None and np.asarray(a).shape == grid_shape

In [23]:
# features = ["sst_anom", "ssl_anom"]
# cols = ["sst", "chlorophyll", "ssl", "cdom_412", "cdom_slope_275_295", "cdom_slope_300_600"]
# keep = df[cols].map(shape_ok).all(axis=1)
# df = df[keep]

# sst_stack   = np.stack(df["sst"].to_numpy()).astype(float)
# chlor_stack = np.stack(df["chlorophyll"].to_numpy()).astype(float)
# ssl_stack   = np.stack(df["ssl"].to_numpy()).astype(float)
# cdom412_stack = np.stack(df["cdom_412"].to_numpy()).astype(float)
# cdom_slope1_stack = np.stack(df["cdom_slope_275_295"].to_numpy()).astype(float)
# cdom_slope2_stack = np.stack(df["cdom_slope_300_600"].to_numpy()).astype(float)

# print(f"Loaded {len(df):,} days, grid shape {grid_shape}")

# def same_day_anomaly(stack):
#     daily_median = np.nanmedian(stack, axis=(1, 2), keepdims=True)
#     return stack - daily_median

# sst_anom_stack = same_day_anomaly(sst_stack)
# ssl_anom_stack = same_day_anomaly(ssl_stack)

In [24]:
blob['data'].columns

Index(['sst', 'sst_n_points', 'chlorophyll', 'chlor_n_points', 'cdom_412',
       'cdom', 'cdom_412_n_points', 'cdom_slope_275_295',
       'cdom_slope_275_295_n_points', 'cdom_slope_300_600',
       'cdom_slope_300_600_n_points', 'ssl', 'ssl_n_points'],
      dtype='object')

In [25]:
bundle = joblib.load(MODEL_PATH)
iforest_pipeline = bundle["iforest_pipeline"]
features = bundle.get("features", features)

preprocess = iforest_pipeline.named_steps["preprocess"]
iforest = iforest_pipeline.named_steps["iforest"]
print(f"Loaded pipeline from {MODEL_PATH}")
print(f"Features: {features}")

Loaded pipeline from F:\Programming\GitHub\nasa-murep\yucatan\experiments\isolation_forest\models\iforest_pipeline4.joblib
Features: ['sst_anom', 'ssl_anom']


In [26]:
start_date = pd.Timestamp("2018-12-01")
end_date = pd.Timestamp("2018-12-30")
percentile_u = 90
percentile_n = 80

In [27]:
window_mask = (df.index >= start_date) & (df.index <= end_date)
n_days = int(window_mask.sum())
if n_days == 0:
    raise ValueError(f"No valid days found between {start_date.date()} and {end_date.date()}")

with np.errstate(invalid="ignore"):
    mean_sst_anom = np.nanmean(sst_anom_stack[window_mask], axis=0)
    mean_ssl_anom = np.nanmean(ssl_anom_stack[window_mask], axis=0)
    mean_chlor    = np.nanmean(chlor_stack[window_mask], axis=0)
    mean_cdom412  = np.nanmean(cdom412_stack[window_mask], axis=0)
    mean_slope1   = np.nanmean(cdom_slope1_stack[window_mask], axis=0)
    mean_slope2   = np.nanmean(cdom_slope2_stack[window_mask], axis=0)

valid_2d = (
    ~np.isnan(mean_sst_anom) & ~np.isnan(mean_ssl_anom)
    & ~np.isnan(mean_chlor) & (mean_chlor > 0)
    & ~np.isnan(mean_cdom412) & ~np.isnan(mean_slope1) & ~np.isnan(mean_slope2)
)
row_w, col_w = np.where(valid_2d)

X_raw = pd.DataFrame({
    "sst_anom": mean_sst_anom[valid_2d],
    "ssl_anom": mean_ssl_anom[valid_2d],
    "chlorophyll": mean_chlor[valid_2d],
    "cdom_412": mean_cdom412[valid_2d],
    "cdom_slope_275_295": mean_slope1[valid_2d],
    "cdom_slope_300_600": mean_slope2[valid_2d],
})

X_window = X_raw[features]
X_window_transformed = preprocess.transform(X_window)

anomaly_score = -iforest.decision_function(X_window_transformed)
threshold_u = np.percentile(anomaly_score, percentile_u)
threshold_n = np.percentile(anomaly_score, percentile_n)
raw_anomaly = anomaly_score >= threshold_u

sst_anom_z = X_window_transformed["anom__sst_anom"].to_numpy()
ssl_anom_z = X_window_transformed["anom__ssl_anom"].to_numpy()

is_upwelling = raw_anomaly & (sst_anom_z < -0.5) & (ssl_anom_z < 0.75)
non_upwelling_mask = anomaly_score < threshold_n

In [28]:
print(f"\n--- Window: {start_date.date()} to {end_date.date()} ({n_days} days averaged) ---")
print(f"Total Valid Pixels: {len(X_raw):,}")
print(f"Upwelling Pixels: {is_upwelling.sum():,} \t Non-Anomaly Pixels: {non_upwelling_mask.sum():,}\n")

print(f"Mean: Upwelling (>{percentile_u}%) vs Non-Anomaly (<{percentile_n}%)")
labels = {
    "sst_anom": "SST anom (°C)",
    "ssl_anom": "SSL anom (m)",
    "chlorophyll": "Chlorophyll (mg/m³)",
    "cdom_412": "CDOM 412",
    "cdom_slope_275_295": "CDOM slope 275-295",
    "cdom_slope_300_600": "CDOM slope 300-600",
}
for col, label in labels.items():
    up_mean = X_raw.loc[is_upwelling, col].mean()
    non_mean = X_raw.loc[non_upwelling_mask, col].mean()
    print(f"{label:22s} | Upwelling: {up_mean:8.4f} | Non-Anomaly: {non_mean:8.4f} | Ratio: {up_mean/non_mean:8.4f}")


--- Window: 2018-12-01 to 2018-12-30 (30 days averaged) ---
Total Valid Pixels: 39,339
Upwelling Pixels: 20 	 Non-Anomaly Pixels: 31,471

Mean: Upwelling (>90%) vs Non-Anomaly (<80%)
SST anom (°C)          | Upwelling:  -1.4580 | Non-Anomaly:  -0.2870 | Ratio:   5.0810
SSL anom (m)           | Upwelling:   0.0712 | Non-Anomaly:  -0.0115 | Ratio:  -6.1965
Chlorophyll (mg/m³)    | Upwelling:   2.4083 | Non-Anomaly:   1.1033 | Ratio:   2.1829
CDOM 412               | Upwelling:   0.1757 | Non-Anomaly:   0.0907 | Ratio:   1.9378
CDOM slope 275-295     | Upwelling:   0.0277 | Non-Anomaly:   0.0320 | Ratio:   0.8659
CDOM slope 300-600     | Upwelling:   0.0213 | Non-Anomaly:   0.0226 | Ratio:   0.9415
